# End-to-End Data Analysis in Python (Healthcare Stroke Dataset)

This notebook walks through a complete **data analysis workflow** in Python using a real-world **healthcare dataset about stroke**.

We will practice:

1. Loading CSV data into pandas  
2. Inspecting and understanding the dataset  
3. Cleaning data (types, missing values, categories)  
4. Exploratory Data Analysis (EDA)  
5. Visualizing distributions, relationships, and correlations  
6. Saving the cleaned dataset

**Dataset**: Healthcare Stroke Prediction Dataset  
Each row represents a patient, with features such as:
- `gender`, `age`, `hypertension`, `heart_disease`, `ever_married`
- `work_type`, `Residence_type`, `avg_glucose_level`, `bmi`, `smoking_status`
- `stroke` (target: 1 = stroke, 0 = no stroke)

---

## How to use this notebook (for learners)

After each section, there is an **Exercises** block.  
Try the exercises *after* running the example code but *before* looking at any solution you might add later.

You can reuse this notebook as a template for other datasets too.


## 1. Setup: Import Libraries

We start by importing the Python libraries we’ll use:

- `numpy` – numerical operations  
- `pandas` – data loading, cleaning, manipulation  
- `matplotlib.pyplot` – plotting library  
- `seaborn` – higher-level interface for statistical plots  

We also configure some display options to make tables and plots easier to read.


In [ ]:
import numpy as np                    # Numerical operations (arrays, math)
import pandas as pd                   # Data loading, manipulation, cleaning
import matplotlib.pyplot as plt       # Base plotting library
import seaborn as sns                 # Statistical visualizations built on Matplotlib

# Make plots appear inside the notebook instead of a separate window
%matplotlib inline

# Optional: prettier display of DataFrames
pd.set_option("display.max_columns", 50)  # Show up to 50 columns
pd.set_option("display.max_rows", 100)    # Show up to 100 rows
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")  # Format floats nicely

# Set a default visual style for seaborn plots
sns.set(style="whitegrid")


### ✅ Exercises – Section 1 (Libraries)

1. Add another pandas display option to show the full width of the notebook (hint: look up `pd.set_option("display.width", ...)`).
2. Change the seaborn style to `"darkgrid"` and re-run any plotting cell later to see the effect.


## 2. Load the Healthcare Stroke Dataset (CSV)

We’ll upload a CSV file from our local machine into Colab, then load it with `pandas.read_csv()`.

Steps:
1. Use `google.colab.files.upload()` to choose the file from your computer.  
2. Ensure the filename in code matches the actual uploaded file name.  
3. Use `pd.read_csv()` to load it into a DataFrame `df`.  
4. Inspect the first few rows with `df.head()`.


In [ ]:
from google.colab import files  # Colab helper functions for file upload/download

print("Please upload the file: healthcare-dataset-stroke-data.csv")
uploaded = files.upload()       # Opens a file picker in the browser

# IMPORTANT: make sure this filename matches the uploaded file.
file_name = "healthcare-dataset-stroke-data.csv"

# read_csv loads the CSV file into a pandas DataFrame.
df = pd.read_csv(file_name)

# Display the first few rows to confirm that the data loaded correctly.
df.head()


### ✅ Exercises – Section 2 (Loading Data)

1. Change `df.head()` to `df.head(10)` to see more rows. What differences do you notice in the data?
2. Use `df.sample(5)` to see 5 random rows. How is this useful compared to `head()`?
3. Try changing the `file_name` to something incorrect and re-run – what error do you see? How does it help you debug file path issues?


## 3. First Look at the Data

We’ll answer basic questions:

- How many rows and columns are there? (`df.shape`)  
- What are the column names? (`df.columns`)  
- What are the data types and how many non-null values does each column have? (`df.info()`)  
- What are basic summary statistics for numeric columns? (`df.describe()`)


In [ ]:
# Number of rows and columns
print("Shape (rows, columns):", df.shape)

# List of column names
print("\nColumn names:")
print(df.columns)

# Data types and non-null counts
print("\nData types and non-null counts:")
df.info()

# Summary stats for numeric columns only
print("\nSummary statistics (numeric columns):")
df.describe()


### ✅ Exercises – Section 3 (Overview)

1. How many rows does your dataset have? How many columns?
2. From `df.info()`, identify:
   - Which columns are `object` (string/categorical)?
   - Which columns are numeric (`int64` or `float64`)?
3. Run `df.describe(include="all")` in a new cell.  
   - What extra information do you get compared to `df.describe()`?


## 4. Inspect Raw Data & Categorical Columns

Sometimes we need to **see actual values** to spot issues like:
- Strange strings (`"N/A"`, `"Unknown"`)
- Inconsistent categories
- Out-of-range values

We’ll use:
- `df.head(n)` – first `n` rows  
- `df.sample(n)` – random `n` rows  
- `df.describe(include="all")` – summary for all columns


In [ ]:
# Look at the first 10 rows
df.head(10)


In [ ]:
# Look at a random sample of 5 rows
df.sample(5, random_state=42)


In [ ]:
# Summary statistics for ALL columns (numeric + non-numeric)
df.describe(include="all")


### ✅ Exercises – Section 4 (Raw Data)

1. Use `df.tail(8)` in a new cell. Do later rows look similar to earlier ones?
2. For the `gender` column, check its unique values using `df["gender"].unique()`.  
   - Do they look clean and consistent?
3. For the `work_type` column, use `df["work_type"].value_counts()`.  
   - Which work type appears most frequently?


## 5. Check Missing Values & Unique Values

Before cleaning, we need to understand:
- Where are missing values?
- Which columns have many unique values (numeric, IDs) vs few (categorical)?

We’ll use:
- `df.isna().sum()` – count missing values per column  
- `df.isna().mean()` – proportion of missing values  
- `df.nunique()` – number of unique values per column


In [ ]:
# Count of missing values per column
print("Missing values per column:")
print(df.isna().sum())

# Percent of missing values per column
print("\nPercentage of missing values per column:")
print((df.isna().mean() * 100).round(2))

# Number of unique values per column
print("\nNumber of unique values per column:")
print(df.nunique())


### ✅ Exercises – Section 5 (Missing & Unique)

1. Which column has the highest percentage of missing values?  
2. For the `bmi` column:
   - Does it have missing values?
   - About what percentage is missing?
3. Look at `df["smoking_status"].nunique()`.  
   - How many smoking categories are there?


## 6. Basic Data Cleaning

We’ll do three things:

1. **Clean column names** – lowercase, remove spaces, replace spaces with `_`  
2. **Fix special missing value codes** – e.g., `"N/A"` in `bmi`  
3. **Convert data types** – make sure numeric columns are truly numeric


In [ ]:
# 1. Clean column names: strip spaces, make lower case, replace spaces with underscores
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

print("Cleaned column names:")
print(df.columns)


In [ ]:
# 2. Inspect unique values in 'bmi' to see special codes or bad values
print(df["bmi"].unique()[:20])  # first 20 unique values


In [ ]:
# Replace the string "N/A" with actual missing values (np.nan)
df["bmi"] = df["bmi"].replace("N/A", np.nan)

# 3. Convert 'bmi' and 'avg_glucose_level' to numeric, coercing bad values to NaN
df["bmi"] = pd.to_numeric(df["bmi"], errors="coerce")
df["avg_glucose_level"] = pd.to_numeric(df["avg_glucose_level"], errors="coerce")

# Check types again
df[["bmi", "avg_glucose_level"]].info()


### ✅ Exercises – Section 6 (Cleaning)

1. Add another column name transformation: replace `"-"` with `"_"` in column names (hint: another `.str.replace("-", "_")` in the chain).
2. Pick another column (e.g., `age`) and ensure it is numeric using `pd.to_numeric`. Does its type change?
3. Create a new column `age_decade` as `df["age"] // 10` (integer division). What type is `age_decade`?


## 7. Handling Missing Values

Now that `bmi` and `avg_glucose_level` are numeric with proper `NaN`s, we can:

- Confirm missing values  
- Impute (fill) them with the **median** of each column  
  (median is robust to outliers)

We’ll use:
- `df["col"].median()` – median value  
- `df["col"].fillna(value)` – fill missing values with `value`


In [ ]:
# Check missing values again after type conversions
df.isna().sum()


In [ ]:
# Compute median for BMI
bmi_median = df["bmi"].median()
print("Median BMI:", bmi_median)

# Fill missing BMI with median
df["bmi"] = df["bmi"].fillna(bmi_median)

# Fill missing avg_glucose_level with its median
agl_median = df["avg_glucose_level"].median()
print("Median Avg Glucose Level:", agl_median)

df["avg_glucose_level"] = df["avg_glucose_level"].fillna(agl_median)

# Confirm missing values after imputation
df.isna().sum()


### ✅ Exercises – Section 7 (Imputation)

1. Count how many missing values `bmi` had **before** and **after** imputation (you may need to rerun `df.isna().sum()` before filling).
2. Try filling `bmi` with the **mean** instead of median in a separate copy of the DataFrame.  
   - What is the difference in the imputed value?
3. For a categorical column like `smoking_status`, think:  
   - Would you use mean/median? If not, what might you use instead (e.g., mode or a special category like `"Unknown"`)? (No code required for this question.)


## 8. Converting Columns to Categorical

Some columns are better treated as **categories** instead of plain strings/integers:

- `gender`, `ever_married`, `work_type`, `residence_type`, `smoking_status`  
- Binary flags: `hypertension`, `heart_disease`, `stroke`

We’ll use:
- `df[col].astype("category")` – convert to pandas categorical type.


In [ ]:
categorical_cols = [
    "gender",
    "ever_married",
    "work_type",
    "residence_type",
    "smoking_status",
    "hypertension",
    "heart_disease",
    "stroke"
]

for col in categorical_cols:
    df[col] = df[col].astype("category")

# Check types of these columns
df[categorical_cols].dtypes


### ✅ Exercises – Section 8 (Categoricals)

1. For `gender`, run `df["gender"].cat.categories`. How many categories are there?
2. For `stroke`, run `df["stroke"].value_counts()`. What proportion of patients have stroke = 1?
3. Convert `age_decade` (from an earlier exercise) to a categorical variable.  
   - Hint: `df["age_decade"] = df["age_decade"].astype("category")`


## 9. Descriptive Statistics – Numeric Columns

We now look at numeric distributions to understand ranges and variability.

Key commands:
- `df.select_dtypes(include=[...])` – select numeric columns  
- `df["col"].describe()` – summary stats for a single column  
- `df.describe()` – summary for all numeric columns


In [ ]:
# Select numeric columns
numeric_df = df.select_dtypes(include=["int64", "float64"])

# Summary statistics for all numeric columns
numeric_df.describe()


In [ ]:
print("Age summary:")
print(df["age"].describe())

print("\nAverage Glucose Level summary:")
print(df["avg_glucose_level"].describe())

print("\nBMI summary:")
print(df["bmi"].describe())


### ✅ Exercises – Section 9 (Numeric Descriptives)

1. Which numeric column has the largest standard deviation? (`std`)
2. What is the minimum and maximum `age` in the dataset?
3. Create a new cell and compute the **interquartile range (IQR)** for `bmi`:  
   - `Q1 = df["bmi"].quantile(0.25)`  
   - `Q3 = df["bmi"].quantile(0.75)`  
   - `IQR = Q3 - Q1`


## 10. Descriptive Statistics – Categorical Columns

For categorical variables, we care about **frequency** and **percentage** distribution.

Key commands:
- `df[col].value_counts()` – raw counts  
- `df[col].value_counts(normalize=True)` – proportions  
We’ll wrap this into a small helper function.


In [ ]:
def show_category_distribution(col_name):
    """
    Print and display the count and percentage of each category for a given column.
    """
    print(f"--- {col_name} ---")
    counts = df[col_name].value_counts()
    percents = df[col_name].value_counts(normalize=True) * 100
    display(pd.DataFrame({"count": counts, "percent": percents.round(2)}))
    print("\n")

for col in ["gender", "ever_married", "work_type", "residence_type", "smoking_status", "stroke"]:
    show_category_distribution(col)


### ✅ Exercises – Section 10 (Categorical Descriptives)

1. Which `work_type` has the highest count?
2. What percentage of people are classified as `"smokes"` in `smoking_status`?
3. Add another column (e.g., `hypertension`) to the loop and inspect its distribution.


## 11. Univariate Visualizations – Numeric

We’ll visualize distributions of:

- `age`
- `avg_glucose_level`
- `bmi`

Using:
- `sns.histplot()` – histogram + optional KDE
- `sns.boxplot()` – boxplot for outlier detection


In [ ]:
# Histograms
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(df["age"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Age Distribution")

sns.histplot(df["avg_glucose_level"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Average Glucose Level Distribution")

sns.histplot(df["bmi"], bins=30, kde=True, ax=axes[2])
axes[2].set_title("BMI Distribution")

plt.tight_layout()
plt.show()


In [ ]:
# Boxplots
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.boxplot(y=df["age"], ax=axes[0])
axes[0].set_title("Age Boxplot")

sns.boxplot(y=df["avg_glucose_level"], ax=axes[1])
axes[1].set_title("Avg Glucose Level Boxplot")

sns.boxplot(y=df["bmi"], ax=axes[2])
axes[2].set_title("BMI Boxplot")

plt.tight_layout()
plt.show()


### ✅ Exercises – Section 11 (Numeric Plots)

1. From the histograms, which variable seems most skewed (not symmetric)?
2. From the boxplots, which variable appears to have the most extreme outliers?
3. Create a **single** histogram of `age` with `bins=15` in a new cell. Add a title and x/y labels.


## 12. Univariate Visualizations – Categorical

We’ll use bar charts (`sns.countplot`) to visualize:

- `gender`
- `work_type`
- `smoking_status`
- `stroke`


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="gender", data=df)
plt.title("Gender Distribution")
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x="work_type", data=df)
plt.title("Work Type Distribution")
plt.xticks(rotation=30)
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x="smoking_status", data=df)
plt.title("Smoking Status Distribution")
plt.xticks(rotation=30)
plt.show()


In [ ]:
plt.figure(figsize=(4, 4))
sns.countplot(x="stroke", data=df)
plt.title("Stroke vs No Stroke")
plt.show()


### ✅ Exercises – Section 12 (Categorical Plots)

1. Which `work_type` category is smallest in the bar chart?
2. What do you notice about the class balance of `stroke` (0 vs 1)?
3. Create a countplot for `residence_type`. Rotate x labels if needed for readability.


## 13. Bivariate Analysis – Stroke vs Risk Factors

Now we explore **relationships**, especially between risk factors and `stroke`.

We’ll use:

- `df.groupby("stroke")["col"].mean()` – compare average values between stroke vs no-stroke  
- `sns.kdeplot()` – compare distribution shapes  
- `sns.boxplot()` – compare distributions & outliers  
- `pd.crosstab()` – cross-tabulate categorical variables


In [ ]:
# Compare mean age, glucose, and BMI for stroke vs no stroke
grouped_stroke = df.groupby("stroke")[["age", "avg_glucose_level", "bmi"]].mean().round(2)
grouped_stroke


In [ ]:
# Age distribution by stroke status
plt.figure(figsize=(8, 4))
sns.kdeplot(data=df, x="age", hue="stroke", shade=True)
plt.title("Age Distribution by Stroke Status")
plt.show()


In [ ]:
# BMI by stroke status (boxplot)
plt.figure(figsize=(6, 4))
sns.boxplot(x="stroke", y="bmi", data=df)
plt.title("BMI by Stroke Status")
plt.show()


In [ ]:
# Cross-tab of hypertension vs stroke
ct_hyper = pd.crosstab(df["hypertension"], df["stroke"], normalize="index") * 100
ct_hyper.round(2)


In [ ]:
# Plot stacked bar for hypertension vs stroke rate
ct_hyper.plot(kind="bar", stacked=True, figsize=(6, 4))
plt.title("Stroke Rate by Hypertension Status (%)")
plt.xlabel("Hypertension (0 = No, 1 = Yes)")
plt.ylabel("Percentage")
plt.legend(title="Stroke", labels=["No Stroke (0)", "Stroke (1)"])
plt.show()


### ✅ Exercises – Section 13 (Bivariate)

1. From `grouped_stroke`, how do average `age` and `avg_glucose_level` differ between stroke and no-stroke groups?
2. In the KDE plot for age, which group tends to be older on average?
3. Create a crosstab similar to `ct_hyper` but for `heart_disease` vs `stroke`.  
   - Plot it as a stacked bar chart.


## 14. Correlation Matrix (Numeric Features)

Correlations help us understand how strongly numeric variables move together.

We’ll use:
- `df.select_dtypes(include=[...]).corr()` – numeric correlation matrix  
- `sns.heatmap()` – visual heatmap of correlations


In [ ]:
# Correlation matrix for numeric columns
corr_matrix = df.select_dtypes(include=["int64", "float64"]).corr()

corr_matrix


In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Matrix of Numeric Features")
plt.show()


### ✅ Exercises – Section 14 (Correlation)

1. Which pair of numeric variables has the strongest positive correlation?
2. Which numeric variable appears most correlated with `age`?
3. Does any variable show a surprisingly low correlation with `stroke` despite being a known risk factor? (Think about reasons why correlation alone might not tell the full story.)


## 15. Save Cleaned Data

After cleaning and exploring, it’s good practice to **save** the cleaned dataset.

We’ll use:
- `df.to_csv("filename.csv", index=False)` – save DataFrame as CSV  
- `files.download("filename.csv")` – download from Colab


In [ ]:
output_file = "healthcare-stroke-data-cleaned.csv"
df.to_csv(output_file, index=False)

print(f"Cleaned data saved to: {output_file}")


In [ ]:
# Download the cleaned file to your computer (Colab only)
files.download(output_file)


### ✅ Final Exercises – Putting It All Together

1. **New Feature Engineering**  
   - Create a new column `high_glucose_flag` which is 1 if `avg_glucose_level` > 140, else 0.  
   - Examine how `high_glucose_flag` is distributed for stroke vs no-stroke patients.

2. **Mini EDA Summary (write-up)**  
   In a markdown cell, write a short summary (4–6 bullet points) describing:
   - What your dataset contains
   - Key patterns you observed (age, BMI, glucose, stroke)
   - Any surprising or interesting findings
   - Possible next steps for modeling (e.g., logistic regression, decision trees)

3. **Bonus – Simple Plot**  
   Create a bar chart showing the average `age` for each `smoking_status` category.

---

You now have a complete notebook template that:
- Loads a healthcare dataset  
- Cleans and inspects it  
- Performs detailed EDA  
- Visualizes important patterns  
- Saves a cleaned version for modeling
